# 4. Geospatial processing

In [ ]:
import ibis
from f_3_spatial import convert_dms_to_decimal
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="build")
db_path = dirs.output_dir / "fame_data.duckdb"
con = ibis.duckdb.connect(str(db_path))
# con.raw_sql("INSTALL spatial; LOAD spatial;")

# DERIVE
table_fame_fixed: ibis.expr.types.Table   = con.table("fame_fixed")
table_fame_derived: ibis.expr.types.Table = con.table("fame_derived")
table_joined = table_fame_fixed.left_join(table_fame_derived, "registered_number")

# Assume you load a free UK Postcode to Lat/Lon lookup CSV into DuckDB
# table_postcode_lookup = con.table("uk_postcodes") 

# 2. Mutate Hierarchy & Coords
table_mutated = table_joined.mutate(
    
    # --- ADDRESS HIERARCHY ---
    # Returns the first option that isn't Null
    best_full_address = ibis.coalesce(
        table_fame_fixed.primary_trading_address,
        table_fame_fixed.ro_address,
        # Fallback: concatenate the separate lines if the above are null
        ibis.literal(", ").join(
            ibis.array([
                table_fame_fixed.ro_address_line_1, 
                table_fame_fixed.ro_address_line_2, 
                table_fame_fixed.ro_address_postcode
            ]).filter(lambda x: x.notnull()) # Only join non-null lines
        )
    ),
    
    # --- GEOSPATIAL HIERARCHY ---
    # 1. Parse FAME's DMS strings into pure decimal floats
    fame_lat_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
    fame_lon_dec = convert_dms_to_decimal(table_fame_fixed.primary_trading_address_longitude),
    
    # 2. (Optional Future Step) If you joined a postcode lookup table, 
    # you would include its lat/lon here as a fallback
    # lookup_lat = table_postcode_lookup.latitude,
    
    # 3. Store the best available coordinates
    best_latitude = ibis.coalesce(
        convert_dms_to_decimal(table_fame_fixed.primary_trading_address_latitude),
        # lookup_lat
    )
)

# 3. Select Derived Schema Columns and Save
table_derived = table_mutated.select(schema_derived_names)
con.create_table("fame_derived", table_derived, overwrite=True)